# Validation robustness grid — CMPE 586

Three fixed checkpoints × clean + 38 existing noisy validation variants. No training or test evaluation. CPU pilot results remain separate. GPU results use fresh GPU clean controls.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT = Path("/content/cmpe586-turkish-punctuation")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "https://github.com/kckagancan/cmpe586-turkish-punctuation.git", str(PROJECT)], check=True)
else:
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=PROJECT, text=True).strip():
        raise RuntimeError("Review checkout changes before updating.")
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT), "-c", str(PROJECT/"requirements/colab.constraints.txt")], check=True)
sys.path.insert(0,str(PROJECT/"src"))
print("Evaluation source:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip())


## Persistent project storage
Connect the existing project Drive if needed. Existing selected models are read from their training directories.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU runtime. This grid must not fall back to CPU.")
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive",timeout_ms=600_000)
STORAGE = Path("/content/drive/MyDrive/cmpe586-turkish-punctuation")
MATRIX = json.loads((PROJECT/"configs/robustness-grid.json").read_text())
GRID = STORAGE/MATRIX["output_directory"]
CALIBRATION = GRID/"calibration.json"
print("GPU:",torch.cuda.get_device_name(0))


## Restore existing validation inputs
Upload the verified validation-inputs-noise-v1.zip into the project inputs folder once. This cell checks transfer hashes and restores existing words/encodings; it does not generate noise.


In [ ]:
from turkish_punctuation.training_io import sha256
ZIP = STORAGE/"inputs/validation-inputs-noise-v1.zip"
INPUTS = Path("/content/cmpe586-evaluation-inputs-noise-v1")
if sha256(ZIP) != MATRIX["input_bundle_sha256"]:
    raise RuntimeError("Input ZIP hash differs.")
if not INPUTS.exists():
    subprocess.run([sys.executable,"scripts/package_evaluation_inputs.py","--verify",str(ZIP),"--sha256",MATRIX["input_bundle_sha256"],"--extract-to",str(INPUTS)],cwd=PROJECT,check=True)
else:
    import zipfile
    manifest=json.loads((INPUTS/"transfer-manifest.json").read_text())
    with zipfile.ZipFile(ZIP) as archive:
        if manifest != json.loads(archive.read("transfer-manifest.json")):
            raise RuntimeError("Cached input manifest differs from the verified ZIP.")
    for name,item in manifest["files"].items():
        path=INPUTS/name
        if path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
            raise RuntimeError(f"Cached input differs: {name}")
    print("Reusing verified existing validation inputs.")
print("Inputs ready; training updates: 0")


## Short GPU timing and memory check
Times 16 fixed representative batches on three input shapes for each model. No validation metrics are computed. Run this once before the grid.


In [ ]:
RUN_CALIBRATION = False
if RUN_CALIBRATION:
    subprocess.run([sys.executable,"-u","scripts/calibrate_evaluation.py","--input-root",str(INPUTS),"--storage-root",str(STORAGE),"--output",str(CALIBRATION)],cwd=PROJECT,check=True)
else:
    print("Calibration is disabled.")
if CALIBRATION.exists():
    report=json.loads(CALIBRATION.read_text())
    print({name:round(value["advisory_grid_minutes"],1) for name,value in report["models"].items()})


## Full GPU evaluation
117 evaluations, then saved-prediction verification, corruption-seed tables and severity figures. Completed condition files are reused after verification on resume. Keep this runtime open; progress/results persist under the project Drive.


In [ ]:
RUN_GRID = False
if RUN_GRID:
    subprocess.run([sys.executable,"-u","scripts/run_robustness_grid.py","--input-root",str(INPUTS),"--storage-root",str(STORAGE),"--calibration",str(CALIBRATION)],cwd=PROJECT,check=True)
else:
    print("Full robustness grid is disabled.")


## Status
The final message is ROBUSTNESS_GRID_COMPLETE. No model/checkpoint deletion is performed.


In [ ]:
for relative in [MATRIX["output_directory"], *[json.loads((PROJECT/name).read_text())["output_directory"] for name in MATRIX["configuration_order"]]]:
    progress=STORAGE/relative/"progress.json"
    if progress.exists():
        print(relative, json.loads(progress.read_text()))

# Read existing committed results; this does not start evaluation.
analysis_directory = STORAGE/MATRIX["output_directory"]/"analysis"
if (analysis_directory/"complete.json").exists():
    from IPython.display import Image, Markdown, display
    publication = json.loads((analysis_directory/"complete.json").read_text())
    for name, expected_digest in publication["files"].items():
        if sha256(analysis_directory/name) != expected_digest:
            raise ValueError(f"Saved analysis file differs: {name}")
    analysis = json.loads((analysis_directory/"analysis.json").read_text())
    print(f"Verified saved report: {analysis['evaluations_verified']} evaluations; held-out test: {analysis['test_evaluated']}")
    display(Markdown((analysis_directory/"results.md").read_text()))
    display(Image(filename=str(analysis_directory/"absolute-f1.png")))
    display(Image(filename=str(analysis_directory/"degradation.png")))
